# Семантический поиск: Sentence Transformers + FAISS

Цель — найти похожие клинические случаи по `description`. Единица поиска — одна строка корпуса `data/processed/dataset_first.csv`; принадлежность строк одному пациенту пока не проверена. `bethesda` и `conclusion` используются только для просмотра найденных случаев, а не для построения векторов. Классификацию Bethesda здесь не решаем.

Ноутбук подготовлен без запуска на реальных данных. До запуска требуется отдельно согласованная проверка технической изоляции и путей. Корпус остаётся неизменным. Индекс хранится в оперативной памяти FAISS; запись индекса на диск и логирование производных медицинских данных здесь отсутствуют. Не сохраняйте ноутбук с выводом реальных данных.

## 1. Условия запуска и зависимости

Выберите ядро локальной `.venv`. `sentence-transformers` и `faiss-cpu` уже перечислены в `requirements.txt`, но их наличие в среде не проверено. Этот ноутбук ничего не устанавливает. Защитный флаг ниже — напоминание, а не техническая изоляция; переключать его можно только после завершения отдельно согласованной проверки.

Модель используется локально: `local_files_only=True` запрещает её автоматическую загрузку. Если весов нет в локальном кеше, остановитесь и отдельно согласуйте подготовку модели. Тексты не передаются в облачный inference API.

In [ ]:
ISOLATION_CONFIRMED = False
if not ISOLATION_CONFIRMED:
    raise RuntimeError(
        'Запуск запрещён до отдельно согласованной проверки технической изоляции.'
    )

from pathlib import Path
from importlib.metadata import version

import faiss
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'requirements.txt').is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError('Не найден корень проекта с requirements.txt.')
    PROJECT_ROOT = PROJECT_ROOT.parent

print({name: version(name) for name in ('sentence-transformers', 'faiss-cpu', 'numpy', 'pandas')})


## 2. Загрузка корпуса

`pd.read_csv` возвращает DataFrame. `usecols` ограничивает чтение тремя нужными колонками: лабораторные номера и другие колонки идентификаторов не загружаются. `sep=';'` и `encoding='utf-8-sig'` повторяют настройки существующего TF-IDF-ноутбука; схема этого CSV агентом не проверялась. `dtype='string'` сохраняет текстовый тип и пропуски. Не выводим исходную таблицу.

Отсутствие колонок идентификаторов не гарантирует отсутствия идентификаторов внутри свободного текста. Перед показом выдачи это нужно проверить локально в рамках разрешённой процедуры.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_first.csv'
RESULT_COLUMNS = ['description', 'bethesda', 'conclusion']
cases = pd.read_csv(
    DATA_PATH, sep=';', encoding='utf-8-sig',
    usecols=RESULT_COLUMNS, dtype='string',
)


## 3. Подготовка описаний

Исключаем пропущенные и пустые описания, убираем только пробелы по краям. Не удаляем отрицания, числа и медицинские термины: они могут менять смысл. `reset_index(drop=True)` задаёт позиции от нуля; вектор с номером `i` будет соответствовать строке `search_cases.iloc[i]`. После построения индекса порядок этой таблицы менять нельзя.

Дубликаты не удаляем автоматически: похожие описания могут принадлежать разным случаям. Их влияние на оценку и повторные случаи одного пациента нужно исследовать отдельно.

In [ ]:
descriptions = cases['description'].str.strip()
valid_description = descriptions.notna() & descriptions.ne('').fillna(False)
search_cases = cases.loc[valid_description, RESULT_COLUMNS].copy()
search_cases['description'] = descriptions.loc[valid_description]
search_cases = search_cases.reset_index(drop=True)
if search_cases.empty:
    raise ValueError('В корпусе нет непустых описаний для поиска.')
print({'index_cases': len(search_cases), 'excluded_empty': len(cases) - len(search_cases)})


## 4. Модель представления текста

Embedding — плотный вектор, представляющий текст. Начинаем с `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`: готовой многоязычной модели для сходства текстов. Она возвращает 384-мерные векторы и не требует обучения на нашем корпусе. Это стартовый вариант, а не подтверждённо лучшая модель для цитологии. Альтернатива — более тяжёлая многоязычная retrieval-модель; сравнивать её нужно на тех же запросах и ручной разметке.

`SentenceTransformer` загружает encoder; `device='cpu'` делает выбор устройства явным. Для воспроизводимого эксперимента укажите в `MODEL_REVISION` commit модели, доступный в локальном кеше. `None` пока означает незакреплённую версию; это ограничение прототипа. [Карточка модели](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2).

Модель обрезает длинные тексты до `max_seq_length` токенов (частей слов), а не символов. Для этой модели штатный предел — 128; увеличение числа без проверки качества не решает проблему автоматически.

In [ ]:
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODEL_REVISION = None  # Перед экспериментом закрепить commit модели.
BATCH_SIZE = 32
model = SentenceTransformer(
    MODEL_NAME, revision=MODEL_REVISION, device='cpu',
    local_files_only=True, trust_remote_code=False,
)
print({'model': MODEL_NAME, 'revision': MODEL_REVISION, 'max_seq_length': model.max_seq_length})


## 5. Векторы корпуса

`model.encode` принимает список описаний и возвращает NumPy-матрицу `(число случаев, размерность)`. `batch_size=32` задаёт число текстов за один проход и влияет на расход памяти. `normalize_embeddings=True` приводит каждый вектор к длине 1; `convert_to_numpy=True` подготавливает результат для FAISS. На этой симметричной модели корпус и запрос кодируем одинаковым методом, без специальных префиксов.

`np.ascontiguousarray(..., dtype=np.float32)` обеспечивает формат памяти, ожидаемый FAISS. Проверяем размеры, конечность чисел и нормировку, не показывая тексты или векторы.

In [ ]:
embeddings = model.encode(
    search_cases['description'].tolist(),
    batch_size=BATCH_SIZE, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(search_cases):
    raise ValueError('Размер матрицы не совпадает с поисковым корпусом.')
if not np.isfinite(embeddings).all():
    raise ValueError('Векторы содержат неконечные значения.')
if not np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5):
    raise ValueError('Векторы должны иметь единичную норму.')
print({'embedding_shape': embeddings.shape})


## 6. Хранение в FAISS

`faiss.IndexFlatIP(d)` создаёт точный индекс для векторов размерности `d`; `add` копирует в него векторы в исходном порядке. Индекс хранит сами векторы, но не описания: тексты берём по возвращённым позициям из `search_cases`.

IP означает скалярное произведение. Для векторов единичной длины оно равно косинусному сходству: больше — похожее. Значение не является вероятностью клинической релевантности. Точный поиск исключает ошибки приближённого индекса при первом эксперименте; альтернатива HNSW полезна позже при ограничениях скорости. [Описание индексов FAISS](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes).

Индекс живёт только в памяти текущего ядра. После перезапуска его нужно построить заново; `write_index` здесь намеренно не вызывается, поскольку отдельное разрешение на медицинские артефакты для этого ноутбука не согласовано.

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
if index.ntotal != len(search_cases):
    raise ValueError('Размер FAISS-индекса не совпадает с корпусом.')
print({'faiss_vectors': index.ntotal, 'dimension': index.d})
del embeddings  # Векторы уже скопированы в FAISS; освобождаем вторую копию.


## 7. Поиск похожих случаев

`find_similar_cases(text, top_k=5)` принимает непустое описание и положительное целое `top_k`, возвращает DataFrame: `rank`, `similarity`, `description`, `bethesda`, `conclusion`. Запрос кодируется той же моделью и нормируется так же, как корпус.

`index.search(query_vector, k)` возвращает две матрицы: оценки и позиции соседей. Ограничиваем `k` числом случаев, чтобы FAISS не возвращал отсутствующие позиции `-1`. Функция ничего не печатает и не записывает. Результаты отсортированы по убыванию сходства; порядок при равных оценках не следует интерпретировать.

Если запрос взят из самого корпуса, собственная строка может оказаться первой. Это допустимо для диагностики, но не для независимой оценки качества. Источник тестовых запросов для этого ноутбука пока не согласован.

In [ ]:
def find_similar_cases(text, top_k=5):
    """Вернуть top-k случаев по косинусному сходству описаний."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError('text должен быть непустой строкой.')
    if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
        raise ValueError('top_k должен быть положительным целым числом.')
    if index.ntotal != len(search_cases) or index.ntotal == 0:
        raise ValueError('Индекс пуст или не соответствует таблице случаев.')

    query_vector = model.encode(
        [text.strip()], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, index.d) or not np.isfinite(query_vector).all():
        raise ValueError('Некорректный вектор запроса.')
    if not np.allclose(np.linalg.norm(query_vector, axis=1), 1.0, atol=1e-5):
        raise ValueError('Вектор запроса должен иметь единичную норму.')

    scores, positions = index.search(query_vector, min(top_k, index.ntotal))
    results = search_cases.iloc[positions[0]][RESULT_COLUMNS].copy()
    results.insert(0, 'similarity', scores[0])
    results.insert(0, 'rank', np.arange(1, len(results) + 1))
    return results.reset_index(drop=True)


## 8. Пример вызова

Ниже — явно вымышленный запрос для знакомства с интерфейсом, не тест качества. Оставляем его пустым по умолчанию, чтобы последовательный запуск не выводил реальный корпус. Для локального просмотра сначала проверьте допустимость вывода свободного текста. Пример синтаксиса: `find_similar_cases('Вымышленное описание: группы клеток и коллоид.', top_k=5)`.

In [ ]:
query_text = ''  # Вставьте отдельно согласованный запрос для локального просмотра.
if query_text.strip():
    results = find_similar_cases(query_text, top_k=5)
    display(results)


## Что эта реализация позволяет проверить

После разрешённой подготовки запуска: сначала на отдельно согласованном synthetic-корпусе проверить соответствие позиций строкам, порядок оценок, `top_k` больше размера корпуса и отклонение пустого запроса. Численные проверки в ячейках контролируют структуру индекса; они не доказывают клиническую релевантность.

Следующий эксперимент требует отдельного согласования. Гипотеза: семантические векторы лучше TF-IDF находят похожие описания при разных формулировках. Baseline — существующий TF-IDF; основной изменяемый фактор — представление текста. Для проверки нужны одни и те же независимые запросы, определение клинической похожести и ручная оценка. Совпадение Bethesda не заменяет релевантность. Для Recall@5/10 нужен эталон релевантных случаев, а не только оценка top-k выдачи. До получения эталона нельзя заявить улучшение.

Перед исследовательским запуском отдельно согласуем учёт в MLflow, критерий принятия модели и допустимые артефакты. Нужно фиксировать участника, дату, версии кода/модели/данных/пакетов, параметры, число случаев индекса и запросов. Здесь MLflow-запуск и метрики не создаются.

**Проверка понимания:** объясните своими словами путь от описания до найденных случаев. Почему для `IndexFlatIP` мы нормируем и корпус, и запрос? Почему после построения индекса нельзя переставлять строки `search_cases`?